In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
import json, os, pathlib, subprocess, sys
REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Content-V9'
EXPECTED_EXACT = 'd9cd6932c3e9532453511203c5a2f5fcbefe8428'
RUNNER_MODULE = 'experiments.run_content_v9_stability'
SOURCE = pathlib.Path('/content/cegwm-content-v9-source')
LOCAL = pathlib.Path('/content/Content-V9-d9cd693-local')
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/CEG-WM/Content')
CAPTURE_LIMIT = 4096


In [ ]:
if SOURCE.exists() or LOCAL.exists(): raise FileExistsError('create-only local path')
subprocess.run(['git','clone','--no-single-branch','--branch',BRANCH,REPO_URL,str(SOURCE)],check=True)
def git(*args): return subprocess.run(['git',*args],cwd=SOURCE,check=True,capture_output=True,text=True).stdout.strip()
if git('branch','--show-current') != BRANCH or git('rev-parse','HEAD') != EXPECTED_EXACT or git('status','--porcelain'): raise RuntimeError('checkout identity')
RUN_UTC = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_TARGET = DRIVE_ROOT / f'Content-V9-{EXPECTED_EXACT[:7]}-{RUN_UTC}'
if DRIVE_TARGET.exists(): raise FileExistsError('create-only Drive target')
subprocess.run([sys.executable,'-m','pip','install',str(SOURCE)],check=True)
if git('branch','--show-current') != BRANCH or git('rev-parse','HEAD') != EXPECTED_EXACT or git('status','--porcelain') or LOCAL.exists() or DRIVE_TARGET.exists(): raise RuntimeError('post-install identity')
from google.colab import userdata
env={k:v for k,v in os.environ.items() if not any(x in k.upper() for x in ('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'))}
env['CEG_WM_ROOT_KEY']=userdata.get('CEG_WM_ROOT_KEY'); env['HF_TOKEN']=userdata.get('HF_TOKEN')
p=subprocess.Popen([sys.executable,'-m',RUNNER_MODULE,'--repo-root',str(SOURCE),'--expected-exact',EXPECTED_EXACT,'--local-work-root',str(LOCAL),'--artifact-sink',str(DRIVE_TARGET)],cwd=SOURCE,env=env,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL)
env.pop('CEG_WM_ROOT_KEY',None); env.pop('HF_TOKEN',None); env=None
captured=p.stdout.read(CAPTURE_LIMIT+1); rc=p.wait()
if len(captured)>CAPTURE_LIMIT or rc not in (0,2): raise RuntimeError('bounded runner result')


In [ ]:
line=captured.decode('utf-8','strict').strip()
print('CEGWM_CONTENT_V9_ARTIFACT '+json.dumps({'execution_exact':EXPECTED_EXACT,'drive_target':str(DRIVE_TARGET),'runner_rc':rc,'result_line':line[:CAPTURE_LIMIT]},sort_keys=True,separators=(',',':')))
